<a href="https://colab.research.google.com/github/mervefilizbaker1/NLP-Homework-02-Word-Embeddings/blob/main/NLP_HMW02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import nbformat

drive.mount('/content/drive')

notebook_path = '/content/drive/MyDrive/NLP-HW2.ipynb'
cleaned_path = '/content/drive/MyDrive/NLP-Hmw02.ipynb'

with open(notebook_path, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

if 'widgets' in nb.metadata:
    del nb.metadata['widgets']
    print("'widgets' key removed from metadata.")
else:
    print("'widgets' key was not found in metadata.")

with open(cleaned_path, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

print(f"Saved: {cleaned_path}")

Mounted at /content/drive
'widgets' key removed from metadata.
Saved: /content/drive/MyDrive/NLP-Hmw02.ipynb


In [ ]:
!pip install datasets

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 49.9 MB/s eta 0:00:00


In [ ]:
!pip install nltk

# Libraries

In [ ]:
from datasets import load_dataset
from gensim.models import Word2Vec
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')
import os

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
import gensim.downloader as api

In [ ]:
from wefe.query import Query
from wefe.word_embedding_model import WordEmbeddingModel
from wefe.metrics import WEAT

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
import numpy as np

# Dataset

In [ ]:
dataset = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

20231101.simple/train-00000-of-00001.par(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/241787 [00:00<?, ? examples/s]

In [ ]:
print(dataset[0]["text"][:200])

April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.

April always begins on the same day 


In [ ]:
print(len(dataset))

241787


# 2.2 Training Word Embeddings

In [ ]:
first_thousand= dataset["text"][:1000]

In [ ]:
all_sentences = []

In [ ]:
for article in first_thousand:
  sentences = sent_tokenize(article)
  for sentence in sentences:
    tokens = [token.lower() for token in word_tokenize(sentence) if token.isalpha()]
    all_sentences.append(tokens)

In [ ]:
print(len(all_sentences))
print(all_sentences[0])

39976
['april', 'apr']


In [ ]:
if os.path.exists("cbow_model.model"):
    os.remove("cbow_model.model")
    print("Removed existing cbow_model.model to prevent incompatibility issues.")

model_cbow = Word2Vec(
    sentences=all_sentences,
    sg= 0,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4
)
model_cbow.save("cbow_model.model")

In [ ]:
if os.path.exists("sgram_model.model"):
    os.remove("sgram_model.model")
    print("Removed existing sgram_model.model to prevent incompatibility issues.")

model_sgram = Word2Vec(
    sentences=all_sentences,
    sg= 1,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4
)
model_sgram.save("sgram_model.model")

In [ ]:
print(model_cbow.wv.most_similar("king"))

[('queen', 0.9542774558067322), ('leader', 0.8744547963142395), ('son', 0.8685175180435181), ('elizabeth', 0.8567734956741333), ('president', 0.8544313907623291), ('independent', 0.8539631366729736), ('monarch', 0.8529831171035767), ('harald', 0.8525572419166565), ('until', 0.8461140394210815), ('army', 0.8436571359634399)]


In [ ]:
print(model_sgram.wv.most_similar("king"))

[('queen', 0.8881316184997559), ('monarch', 0.8679859042167664), ('prince', 0.8643187284469604), ('son', 0.8603602051734924), ('mary', 0.8440443277359009), ('edward', 0.8390122056007385), ('vi', 0.8377383947372437), ('married', 0.837132453918457), ('elizabeth', 0.8347997665405273), ('emperor', 0.8314002752304077)]


In [ ]:
all_articles = dataset["text"]

In [ ]:
for article in all_articles:
  sentences = sent_tokenize(article)
  for sentence in sentences:
    tokens = [token.lower() for token in word_tokenize(sentence) if token.isalpha()]
    all_sentences.append(tokens)

In [ ]:
print(len(all_sentences))
print(all_sentences[0])

2425742
['april', 'apr']


# 2.3 Comparing Word Embeddings

In [ ]:
glove = api.load("glove-wiki-gigaword-100")
google = api.load("word2vec-google-news-300")

[==================================================] 100.0% 128.1/128.1MB downloaded
[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
glove_100 = api.load("glove-wiki-gigaword-100")
glove_300 = api.load("glove-wiki-gigaword-300")

[==================================================] 100.0% 376.1/376.1MB downloaded


In [ ]:
queries = [
    {"type": "similarity", "query": "guitar"},
    {"type": "similarity", "query": "actor"},
    {"type": "analogy", "positive": ["ankara", "germany"], "negative": ["turkey"]},
    {"type": "analogy", "positive": ["mother", "man"], "negative": ["woman"]},
    {"type": "analogy", "positive": ["algorithm", "biology"], "negative": ["computer"]},
]

In [ ]:
models = {
    "CBOW": model_cbow.wv,
    "Skip-gram": model_sgram.wv,
    "GloVe-100": glove_100,
    "GloVe-300": glove_300
}

for name, model in models.items():
    for q in queries:
        if q["type"] == "similarity":
            result = model.most_similar(q["query"])
        else:
            result = model.most_similar(positive=q["positive"], negative=q["negative"])
        print(f"\n{name} | {q['type']}: {q.get('query', q.get('positive'))}")
        print(result)


CBOW | similarity: guitar
[('compass', 0.9736208319664001), ('mail', 0.9724996089935303), ('aircraft', 0.9717581272125244), ('redshift', 0.9704495072364807), ('utility', 0.9689320921897888), ('leaf', 0.9686363935470581), ('parody', 0.9681756496429443), ('driver', 0.9676082134246826), ('setting', 0.9672145843505859), ('cherubim', 0.9668338894844055)]

CBOW | similarity: actor
[('actress', 0.9952216744422913), ('footballer', 0.9866959452629089), ('singer', 0.9844109416007996), ('musician', 0.9822180867195129), ('comedian', 0.9816185832023621), ('politician', 0.9760420322418213), ('drummer', 0.974768340587616), ('writer', 0.9741107225418091), ('director', 0.9717271327972412), ('boyer', 0.9709035754203796)]

CBOW | analogy: ['ankara', 'germany']
[('constitutional', 0.9601251482963562), ('la', 0.9368729591369629), ('interim', 0.93623286485672), ('ceylon', 0.9313110113143921), ('vancouver', 0.9291106462478638), ('federal', 0.9276981949806213), ('federative', 0.9271276593208313), ('canton', 

# 2.4 Bias in Word Embeddings

In [ ]:
!pip install wefe

In [ ]:
christianity_words = ["christian", "church", "bible", "jesus", "christmas", "cross", "prayer", "cathedral"]
islam_words = ["muslim", "mosque", "quran", "allah", "ramadan", "imam", "islam", "mecca"]

In [ ]:
positive_words = ["love", "peace", "joy", "happy", "wonderful", "beautiful", "good", "pleasant"]
negative_words = ["war", "terror", "evil", "bad", "horrible", "violent", "terrible", "hate"]

In [ ]:
query = Query(
    target_sets=[christianity_words, islam_words],
    attribute_sets=[positive_words, negative_words],
    target_sets_names=["Christianity", "Islam"],
    attribute_sets_names=["Positive", "Negative"]
)

In [ ]:
models = [
    WordEmbeddingModel(model_cbow.wv, "CBOW"),
    WordEmbeddingModel(model_sgram.wv, "Skip-gram"),
    WordEmbeddingModel(glove_100, "GloVe-100"),
    WordEmbeddingModel(glove_300, "GloVe-300"),
]

In [ ]:
for model in models:
    result = WEAT().run_query(query, model, lost_vocabulary_threshold=0.5)
    print(f"\n{model.name}")
    print(result)


CBOW
{'query_name': 'Christianity and Islam wrt Positive and Negative', 'result': np.float64(-0.13351030094282923), 'weat': np.float64(-0.13351030094282923), 'effect_size': np.float64(-0.4510015236031692), 'p_value': nan}

Skip-gram
{'query_name': 'Christianity and Islam wrt Positive and Negative', 'result': np.float64(0.21643991470336932), 'weat': np.float64(0.21643991470336932), 'effect_size': np.float64(0.9399827843666135), 'p_value': nan}

GloVe-100
{'query_name': 'Christianity and Islam wrt Positive and Negative', 'result': np.float64(1.1540158810094), 'weat': np.float64(1.1540158810094), 'effect_size': np.float64(1.7685541244018692), 'p_value': nan}

GloVe-300
{'query_name': 'Christianity and Islam wrt Positive and Negative', 'result': np.float64(1.05700358573813), 'weat': np.float64(1.05700358573813), 'effect_size': np.float64(1.673571685328266), 'p_value': nan}


# 2.5 Text Classification: Sparse vs. Dense

In [ ]:
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
{'text': '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', 'label': 2}


In [ ]:
train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]
test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

In [ ]:
vectorizer = TfidfVectorizer(max_features=50000)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train,train_labels)

LogisticRegression(max_iter=1000)

In [ ]:
preds = clf.predict(X_test)
print(classification_report(test_labels, preds))

              precision    recall  f1-score   support

           0       0.67      0.35      0.46      3972
           1       0.59      0.76      0.66      5937
           2       0.54      0.58      0.56      2375

    accuracy                           0.59     12284
   macro avg       0.60      0.56      0.56     12284
weighted avg       0.61      0.59      0.58     12284



In [ ]:
def get_avg_embedding(text, model, vector_size):
    tokens = text.lower().split()
    vectors = []
    for token in tokens:
        if token in model:
            vectors.append(model[token])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size)

In [ ]:
embedding_models = {
    "CBOW": (model_cbow.wv, 100),
    "Skip-gram": (model_sgram.wv, 100),
    "GloVe-100": (glove_100, 100),
    "GloVe-300": (glove_300, 300),
}

In [ ]:
for name, (emb_model, vec_size) in embedding_models.items():

    X_train_emb = np.array([get_avg_embedding(t, emb_model, vec_size) for t in train_texts])
    X_test_emb = np.array([get_avg_embedding(t, emb_model, vec_size) for t in test_texts])

    clf_emb = LogisticRegression(max_iter=1000)
    clf_emb.fit(X_train_emb, train_labels)

    preds_emb = clf_emb.predict(X_test_emb)
    print(f"\n Model B — {name}")
    print(classification_report(test_labels, preds_emb))


 Model B — CBOW
              precision    recall  f1-score   support

           0       0.45      0.07      0.13      3972
           1       0.52      0.58      0.55      5937
           2       0.26      0.55      0.36      2375

    accuracy                           0.41     12284
   macro avg       0.41      0.40      0.34     12284
weighted avg       0.45      0.41      0.37     12284


 Model B — Skip-gram
              precision    recall  f1-score   support

           0       0.48      0.13      0.21      3972
           1       0.53      0.64      0.58      5937
           2       0.32      0.53      0.40      2375

    accuracy                           0.46     12284
   macro avg       0.44      0.43      0.39     12284
weighted avg       0.47      0.46      0.42     12284


 Model B — GloVe-100
              precision    recall  f1-score   support

           0       0.59      0.52      0.55      3972
           1       0.61      0.59      0.60      5937
           2  